# Lecture 4 — What does `fit()` actually do?

**Companion notebook for Frankfurt UAS · Machine Learning · Lecture 4 of 9**

This notebook walks through two model families on the same AAPL forecasting task:

1. **Linear Regression** — a parametric model. `fit()` learns four numbers.
2. **KNN** — an instance-based model. `fit()` just stores the data.

Each section maps to a block in the slides. Run cells top-to-bottom during the lecture.

---

## Setup

We need pandas, numpy, matplotlib, and scikit-learn. Optionally yfinance for live data.
If yfinance fails (offline / API issues), the notebook falls back to synthetic data with
the same structure — so you can run everything either way.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa — registers 3D projection
import time

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

# Cosmetic
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
np.random.seed(42)

print("All imports OK.")

---

## Section 1 — Setup & data

We download AAPL closing prices and build three features per trading day:

- **Close** today's closing price
- **Volume** today's trading volume
- **MA7** the 7-day moving average

The **target** `y` is **tomorrow's closing price** — known historically, unknown for new days.

In [ ]:
# Try to load real AAPL data; fall back to synthetic data if offline.
def load_real_aapl():
    import yfinance as yf
    df = yf.download("AAPL", start="2022-01-01", end="2024-12-31",
                     progress=False, auto_adjust=True)
    if df.empty:
        raise RuntimeError("yfinance returned no data")
    # Flatten columns if MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df[["Close", "Volume"]].copy()

def make_synthetic_aapl():
    """Synthetic stock-like series with same structure as real AAPL."""
    n = 750  # ~3 years of trading days
    dates = pd.bdate_range("2022-01-03", periods=n)
    # Random walk for price, with mild trend
    returns = np.random.randn(n) * 0.015 + 0.0004
    price = 150 * np.exp(np.cumsum(returns))
    volume = np.random.gamma(2.0, 50_000_000 / 2.0, size=n)
    df = pd.DataFrame({"Close": price, "Volume": volume}, index=dates)
    return df

try:
    df = load_real_aapl()
    source = "yfinance (live)"
except Exception as e:
    print(f"⚠️  yfinance unavailable ({type(e).__name__}); using synthetic data.")
    df = make_synthetic_aapl()
    source = "synthetic"

print(f"Data source: {source}")
print(f"Rows: {len(df)}  ·  date range: {df.index.min().date()} → {df.index.max().date()}")
df.head()

Now we build the feature `MA7` and the target `y` (tomorrow's close):

In [ ]:
df["MA7"] = df["Close"].rolling(7).mean()
df["target"] = df["Close"].shift(-1)   # tomorrow's close
df = df.dropna()

X = df[["Close", "Volume", "MA7"]]
y = df["target"]

print(f"X shape: {X.shape}   ·   features: {list(X.columns)}")
print(f"y shape: {y.shape}   ·   target: tomorrow's Close")
print()
X.head()

**That's the universal schema, made concrete:**

- **Input X** = three columns above
- **Target y** = tomorrow's close (known for our historical data)
- **Output ŷ** = whatever the model will predict, once we fit one

Next we'll fit a model. But first, let's *see* the feature space.

---

## Section 2 — The feature space

Each trading day is a single **point in 3D space**, located by its (Close, Volume, MA7).
The colour shows tomorrow's price — that's what the model will try to predict from position.

When we say a model "learns", we mean it figures out the relationship between a point's
position and its colour. That is **Phase 2** — interpretation of the position.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(X["Close"], X["Volume"], X["MA7"],
                c=y, cmap="viridis", s=20, alpha=0.7)
ax.set_xlabel("Close today")
ax.set_ylabel("Volume")
ax.set_zlabel("MA7")
plt.colorbar(sc, label="target (tomorrow's close)")
plt.title("AAPL feature space — each point is a trading day")
plt.tight_layout()
plt.show()

You should see structure: points with similar features tend to have similar colours.
That structure is what makes prediction possible. If colours were scattered randomly,
no model would have anything to learn from.

---

## Section 3 — Family 1: Linear Regression

Linear Regression is a **parametric** model. The model is the formula:

$$\hat{y} = w_1 \cdot \text{Close} + w_2 \cdot \text{Volume} + w_3 \cdot \text{MA7} + b$$

Four numbers — $w_1, w_2, w_3, b$. These are the **parameters** $\theta$.
`fit()` will find good values for them.

**First: split with TimeSeriesSplit, never random shuffle for stock data.**

In [ ]:
# Time-series split: train on older days, test on newer ones.
# Random shuffling would leak the future into the training set.
tscv = TimeSeriesSplit(n_splits=5)
train_idx, test_idx = list(tscv.split(X))[-1]  # use the last (most realistic) fold
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"Train: {len(X_train)} days   ·   {X_train.index[0].date()} → {X_train.index[-1].date()}")
print(f"Test:  {len(X_test)} days   ·   {X_test.index[0].date()} → {X_test.index[-1].date()}")

**Now fit. Pause and ask yourself: what is `fit()` about to do?**

In [ ]:
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
print("Fit complete.")

**The trained model lives in four numbers. Let's look at them.**

In [ ]:
print("Coefficients (θ):")
for name, w in zip(X.columns, model_lr.coef_):
    print(f"   {name:8s}  =  {w:+.6f}")
print(f"\nIntercept (b)  =  {model_lr.intercept_:+.4f}")
print()
print(">>> These four numbers ARE the trained model.")
print(">>> Save them, throw the data away, and you can still predict.")

In [ ]:
# Predictions on the test set
y_pred_lr = model_lr.predict(X_test)
mse_lr = np.mean((y_test - y_pred_lr)**2)
print(f"MSE on test:  {mse_lr:.4f}")
print(f"RMSE on test: {np.sqrt(mse_lr):.4f}  (in price units)")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(12, 4))
plt.plot(y_test.index, y_test.values, label="actual", linewidth=2)
plt.plot(y_test.index, y_pred_lr, label="LR prediction", linestyle="--")
plt.title("Linear Regression — AAPL forecast on test set")
plt.ylabel("Close (next day)")
plt.legend()
plt.tight_layout()
plt.show()

The model fits the broad trend but misses the short-term wiggles. That's expected —
LR is a simple linear model with three features. More features and a more flexible model
would help, but the *mechanism* of `fit()` is what matters here, not absolute accuracy.

> **Mentimeter Poll #2 (in lecture):** What did `fit()` change in our LR model?
> (a) The training data
> (b) The four numbers $w_1, w_2, w_3, b$
> (c) The number of features
> (d) The loss function
>
> *(Answer: b)*

---

## Section 4 — Family 2: KNN

KNN is **instance-based**. The model is *the training data itself*.
There is no formula with knobs. To predict, KNN computes the distance to all training
points, takes the k closest, and averages their targets.

**One critical step before fitting KNN: scale the features.**

KNN uses Euclidean distance. If `Volume` ranges 0–100 million and `Close` ranges 100–200,
distance is dominated by `Volume` — `Close` becomes irrelevant. Always scale for KNN.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Original feature ranges:")
print(X_train.describe().loc[["min", "max"]].round(2))
print()
print("Scaled feature ranges:")
print(pd.DataFrame(X_train_scaled, columns=X.columns).describe().loc[["min", "max"]].round(2))

**Now fit. What's `fit()` about to do this time?**

In [ ]:
model_knn = KNeighborsRegressor(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)
print("Fit complete.")

**There's no `coef_` to inspect. Let's prove it.**

In [ ]:
try:
    print(model_knn.coef_)
except AttributeError as e:
    print(f"AttributeError: {e}")
    print()
    print(">>> KNN has no coefficients. There are no parameters to learn.")

In [ ]:
# What KNN.fit() actually did was: store the data.
print(f"Stored training data shape:  {model_knn._fit_X.shape}")
print(f"Stored training targets:     {model_knn._y.shape}")
print()
print(">>> The 'model' for KNN is just a reference to the training data.")
print(">>> fit() saved that reference. That's it.")

**Now vary `k` and see how the test error changes.**

`k` is a *hyperparameter* — set by us, not learned by fit().

In [ ]:
results = []
for k in [1, 3, 5, 10, 25, 50, 100]:
    m = KNeighborsRegressor(n_neighbors=k).fit(X_train_scaled, y_train)
    pred = m.predict(X_test_scaled)
    mse = np.mean((y_test - pred)**2)
    results.append((k, mse))
    print(f"k = {k:3d}   ·   MSE = {mse:.4f}   ·   RMSE = {np.sqrt(mse):.4f}")

# Plot the curve
ks, mses = zip(*results)
plt.figure(figsize=(8, 4))
plt.plot(ks, mses, marker="o")
plt.xlabel("k (number of neighbors)")
plt.ylabel("test MSE")
plt.title("Bias-variance tradeoff in KNN, via k")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

- `k=1`: the model follows every training point exactly — wobbly, **overfit**.
- Large `k`: the model averages over a huge neighborhood — smooth but unresponsive, **underfit**.
- Sweet spot is in between.

This is bias-variance, made concrete. We'll formalise it in V6.

In [ ]:
# Predictions for the chosen k=5
y_pred_knn = model_knn.predict(X_test_scaled)
plt.figure(figsize=(12, 4))
plt.plot(y_test.index, y_test.values, label="actual", linewidth=2)
plt.plot(y_test.index, y_pred_knn, label="KNN prediction (k=5)", linestyle="--")
plt.title("KNN — AAPL forecast on test set")
plt.ylabel("Close (next day)")
plt.legend()
plt.tight_layout()
plt.show()

---

## Section 5 — Side-by-side: two stories of `fit()`

Now we compare LR and KNN directly. Same data, same task. Two completely different stories
of what `fit()` did.

**First: timing. Where does each model do its work?**

In [ ]:
# Time fit() — KNN should be near-instant, LR slightly slower (but still fast)
t0 = time.perf_counter()
LinearRegression().fit(X_train, y_train)
t_lr_fit = time.perf_counter() - t0

t0 = time.perf_counter()
KNeighborsRegressor(n_neighbors=5).fit(X_train_scaled, y_train)
t_knn_fit = time.perf_counter() - t0

# Time predict()
m_lr = LinearRegression().fit(X_train, y_train)
m_knn = KNeighborsRegressor(n_neighbors=5).fit(X_train_scaled, y_train)

# Run predict multiple times for a stable measurement
N_REPEATS = 100
t0 = time.perf_counter()
for _ in range(N_REPEATS):
    m_lr.predict(X_test)
t_lr_pred = (time.perf_counter() - t0) / N_REPEATS

t0 = time.perf_counter()
for _ in range(N_REPEATS):
    m_knn.predict(X_test_scaled)
t_knn_pred = (time.perf_counter() - t0) / N_REPEATS

print(f"{'':12s}  {'fit (ms)':>12s}  {'predict (ms)':>14s}")
print(f"{'-'*12}  {'-'*12}  {'-'*14}")
print(f"{'LR':12s}  {t_lr_fit*1000:>12.3f}  {t_lr_pred*1000:>14.3f}")
print(f"{'KNN':12s}  {t_knn_fit*1000:>12.3f}  {t_knn_pred*1000:>14.3f}")
print()
print(">>> LR works during fit. KNN works during predict.")
print(">>> The total work is similar — it just lives in a different place.")

**Second: predictions side by side.**

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(y_test.index, y_test.values, label="actual", linewidth=2.5, color="black")
plt.plot(y_test.index, m_lr.predict(X_test), label="LR", linestyle="--", linewidth=1.5)
plt.plot(y_test.index, m_knn.predict(X_test_scaled), label="KNN (k=5)", linestyle=":", linewidth=1.5)
plt.title("LR vs KNN — same data, two different worlds")
plt.ylabel("Close (next day)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Final comparison table
mse_lr = np.mean((y_test - m_lr.predict(X_test))**2)
mse_knn = np.mean((y_test - m_knn.predict(X_test_scaled))**2)

summary = pd.DataFrame({
    "Model": ["LR", "KNN"],
    "fit time (ms)": [round(t_lr_fit*1000, 3), round(t_knn_fit*1000, 3)],
    "predict time (ms)": [round(t_lr_pred*1000, 3), round(t_knn_pred*1000, 3)],
    "test MSE": [round(mse_lr, 4), round(mse_knn, 4)],
    "test RMSE": [round(np.sqrt(mse_lr), 4), round(np.sqrt(mse_knn), 4)],
    "Has parameters?": ["yes (4 numbers)", "no"],
    "Has hyperparameters?": ["yes", "yes (k)"],
    "Knowledge lives in": ["θ — the four numbers", "the training data itself"],
})
summary.set_index("Model")

> **Mentimeter Poll #3 (end of lecture):** What happens when you call `model.fit(X, y)`?
>
> Compare your one-word warm-up answer with what you'd say now.
> The schema gives you the structure; the family gives you the content.
>
> - **For LR:** `fit()` finds four numbers that minimise MSE.
> - **For KNN:** `fit()` stores the training data. The work is deferred to predict time.

---

## Section 6 — Bonus / homework

Three exercises to deepen the lecture content. The cells below are starters —
fill in the missing pieces yourself.

### Exercise 1 — Add a fourth feature

Add another feature to `X`, e.g. RSI or price volatility (rolling std). Refit both LR and KNN.
How does each model react?

In [ ]:
# Starter:
df["volatility"] = df["Close"].rolling(7).std()
df_ex = df.dropna().copy()

X_ex = df_ex[["Close", "Volume", "MA7", "volatility"]]
y_ex = df_ex["target"]

# TODO: split, fit LR and KNN with the new feature set, compare to the previous results.
# Hint: did adding the feature help LR more than KNN, or the other way around?

### Exercise 2 — KNN with k=1

Set `k=1` and look at predictions on the **training set**. What do you see — and why?

In [ ]:
# Starter:
m_overfit = KNeighborsRegressor(n_neighbors=1).fit(X_train_scaled, y_train)
y_pred_train = m_overfit.predict(X_train_scaled)

train_mse = np.mean((y_train - y_pred_train)**2)
print(f"k=1 train MSE: {train_mse:.6f}")
# TODO: explain in one sentence WHY this number is what it is.

### Exercise 3 — A simple "trading rule"

Both LR and KNN predict tomorrow's *price*. Convert that into a trading signal:
buy if the prediction is higher than today's close, sell otherwise. Compute the hit rate
(how often you were directionally right).

In [ ]:
# Starter:
predictions_lr = m_lr.predict(X_test)
today_close = X_test["Close"].values
actual_tomorrow = y_test.values

predicted_direction = predictions_lr > today_close
actual_direction = actual_tomorrow > today_close
hit_rate = np.mean(predicted_direction == actual_direction)
print(f"LR directional hit rate: {hit_rate:.1%}")
# TODO: do the same for KNN. Is one of them clearly better at direction?
# Bonus: 50% is random guessing — interpret your number.

---

## Wrap-up

By the end of this notebook you've seen — concretely, with numbers — that:

1. **Both models follow the universal schema** Input → Model → Output → compare to target.
2. **`fit()` does completely different things** for LR and KNN.
   - LR's `fit()` produced four numbers (`coef_`, `intercept_`).
   - KNN's `fit()` stored the training data — there is no `coef_`.
3. **The work shifts** between training and prediction time, depending on the family.
4. **Both are "machine learning"** — neither is the right or wrong answer.

Next lecture (V5): the **Map of ML** — where every model fits in a single landscape.